# MACHINE LEARNING MODELS

## SETTING UP

### Imports

In [52]:
%pip install lightgbm xgboost catboost scikit-learn pickle-blosc

   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------- ----------------- 1.0/1.9 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------- 1.9/1.9 MB 7.5 MB/s  0:00:00

   ---------------- ----------------------- 2/5 [blosc]
   ------------------------ --------------- 3/5 [pytest]
   ------------------------ --------------- 3/5 [pytest]
   ------------------------ --------------- 3/5 [pytest]
   ------------------------ --------------- 3/5 [pytest]
   ------------------------ --------------- 3/5 [pytest]
   ------------------------ --------------- 3/5 [pytest]
   ------------------------ --------------- 3/5 [pytest]
   ---------------------------------------- 5/5 [pickle-blosc]

Note: you may need to restart the kernel to use updated packages.


In [53]:
import pandas as pd
import numpy as np
import sklearn.metrics as metrics
import os
from pickle_blosc import pickle, unpickle
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, TargetEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, train_test_split

###  DataLoader Class

In [6]:
class DiabeticDataLoader:
    def __init__(self, df_raw):
        self.df_raw = df_raw.copy()
        self.df_clean = None
        self.df_no_outliers = None
        self.scaler = None

    def _map_icd9(self, val):
        """Mapeia os códigos CID-9 para as 9 macro categorias presentes no artigo sobre esse Dataset."""
        if pd.isna(val) or val == '?':
            return 'Missing'

        val_str = str(val)
        # Códigos suplementares (V e E) entram em 'Other'
        if val_str.startswith('V') or val_str.startswith('E'):
            return 'Other'

        try:
            # Pega apenas a raiz do diagnóstico antes do ponto
            num = float(val_str.split('.')[0])

            if num == 250:
                return 'Diabetes'
            elif (390 <= num <= 459) or num == 785:
                return 'Circulatory'
            elif (460 <= num <= 519) or num == 786:
                return 'Respiratory'
            elif (520 <= num <= 579) or num == 787:
                return 'Digestive'
            elif 800 <= num <= 999:
                return 'Injury'
            elif 710 <= num <= 739:
                return 'Musculoskeletal'
            elif (580 <= num <= 629) or num == 788:
                return 'Genitourinary'
            elif 140 <= num <= 239:
                return 'Neoplasms'
            else:
                return 'Other'
        except ValueError:
            return 'Other'

    def clean_data(self):
        df = self.df_raw.copy()

        # 1. Remover colunas desnecessárias, com alta porcentagem de vazios, ou constantes avaliados pelo artigo
        constant_cols = ['examide', 'citoglipton', 'weight', 'payer_code', 'encounter_id', 'patient_nbr', 'weight', 'medical_specialty']
        df.drop(columns=constant_cols, inplace=True, errors='ignore')

        # 2. Tratamento de valores faltantes e marcações nulas
        df.replace('?', np.nan, inplace=True)
        for col in ['max_glu_serum', 'A1Cresult']:
            df[col] = df[col].replace('None', np.nan)

        # 3. Agregação CID-9
        for col in ['diag_1', 'diag_2', 'diag_3']:
            if col in df.columns:
                df[col] = df[col].apply(self._map_icd9)

        # 4. Redução da Dimensionalidade das colunas de medicamentos
        med_cols = [
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
            'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
            'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
            'insulin', 'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
        ]
        df['total_meds_up'] = (df[med_cols] == 'Up').sum(axis=1)
        df['total_meds_down'] = (df[med_cols] == 'Down').sum(axis=1)
        df['total_meds_steady'] = (df[med_cols] == 'Steady').sum(axis=1)
        df.drop(columns=med_cols, inplace=True, errors='ignore')

        # 5. Redução da Dimensionalidade das colunas de contagem de visitas anteriores do paciente
        visit_cols = ['number_outpatient', 'number_emergency', 'number_inpatient']
        df['total_prior_visits'] = df[visit_cols].sum(axis=1)
        df.drop(columns=visit_cols, inplace=True, errors='ignore')

        # 6. Atualização de tipos para Categóricos
        categorical_cols = [
            'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id', 'medical_specialty',
            'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult',
            'change', 'diabetesMed', 'readmitted'
        ]
        # for col in categorical_cols:
        #     df[col] = df[col].fillna('Missing')
        #     df[col] = df[col].astype('category')

        # TESTE
        for col in df.columns:
            if df[col].isnull().sum() > 0:
                if not pd.api.types.is_numeric_dtype(df[col]):
                    df[col] = df[col].fillna(df[col].mode()[0], inplace=True)
                else:
                    df[col] = df[col].fillna(df[col].median(), inplace=True)

        # DataFrame Limpo e com Dimensão Reduzida
        self.df_clean = df
        return df

    def remove_outliers(self, df=None):
        if df is None:
            if self.df_clean is None:
                raise ValueError("Data must be cleaned before outlier removal.")
            else:
                df = self.df_clean.copy()

        outlier_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'total_prior_visits'
        ]

        self.thresholds = {}
        for col in outlier_cols:
            if col in df.columns:
                low = df[col].min()
                high = df[col].quantile(0.99)
                self.thresholds[col] = (low, high)

        mask = pd.Series(True, index=df.index)
        for feature, (low, high) in self.thresholds.items():
            if feature in df.columns:
                mask &= df[feature].between(low, high)

        df_no_outliers = df.loc[mask].copy()
        self.df_no_outliers = df_no_outliers
        return df_no_outliers

    def get_clean_data(self):
        if self.df_clean is None:
            self.clean_data()
        return self.df_clean

    def get_no_outliers_data(self):
        if self.df_no_outliers is None:
            self.remove_outliers()
        return self.df_no_outliers

    def get_features_target(self):
        df = self.get_clean_data()
        exclude_cols = ['encounter_id', 'patient_nbr', 'readmitted']
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        X = df[feature_cols]
        y = df['readmitted']
        return X, y

    def get_train_test_split(self, test_size=0.2, random_state=42):
        df_clean = self.clean_data()
        X, y = self.get_features_target()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=random_state
        )

        train_combined = pd.concat([X_train, y_train], axis=1)
        train_combined = self.remove_outliers(train_combined)

        X_train = train_combined.drop(columns=['readmitted'])
        y_train = train_combined['readmitted']

        numeric_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'total_prior_visits', 'num_procedures', 'number_diagnoses',
            'total_meds_up', 'total_meds_down', 'total_meds_steady'
        ]

        self.scaler = StandardScaler()
        X_train[numeric_cols] = self.scaler.fit_transform(X_train[numeric_cols])
        X_test[numeric_cols] = self.scaler.transform(X_test[numeric_cols])

        return X_train, X_test, y_train, y_test

#### How to Use It

| # | Method                       | Description                                |
|---|------------------------------|--------------------------------------------|
| 1 | `get_clean_data()`           | Full cleaned dataframe                     |
| 1 | `get_outlier_treated_data()` | Flagging  outliers                         |
| 1 | `get_no_outliers_data()`     | Removing outliers                          |
| 1 | `standardize_features()`     | Standardizing features                     |
| 2 | `get_features_target()`      | Modeling features and target (no outliers) |
| 3 | `get_train_test_split()`     | Reproducible encoded train/test split      |

### DataTrainer Class

In [7]:
class DiabeticDataTrainer:
    def __init__ (self, df_raw):
        self.df_raw = df_raw
        self.df_diabetic_loader = DiabeticDataLoader(df_raw)
        self.df_clean = None
        self.X_test = None
        self.y_test_raw = None
        self.X_train = None
        self.y_train_raw = None
        self.y_test_encoded = None
        self.y_train_encoded = None
        self.preprocessor = None
        self.label_encoder = None

        self.__set_up_data()
        self.__set_up_preprocessor()

    def __set_up_data(self):
        # Limpando dados
        self.df_clean = self.df_diabetic_loader.clean_data()

        # Fazendo o split
        self.X_train, self.X_test, self.y_train_raw, self.y_test_raw = self.df_diabetic_loader.get_train_test_split()

        # Fazendo o encoding do y
        self.label_encoder = LabelEncoder()
        self.y_train_encoded = self.label_encoder.fit_transform(self.y_train_raw)
        self.y_test_encoded = self.label_encoder.transform(self.y_test_raw)

    def __set_up_preprocessor(self):
        # Colunas categóricas com muitas categorias ordinais (usa OrdinalEncoder)
        ordinal_cols = [
            'age'
        ]

        # Colunas categóricas com muitas categorias não-ordinais (usa TargetEncoder)
        target_cols = [
            'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
            'diag_1', 'diag_2', 'diag_3'
        ]

        # Colunas numéricas (usa StandardScaler)
        numerical_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'total_prior_visits', 'num_procedures', 'number_diagnoses'
        ]

        # Colunas categóricas com poucas categorias (usa OneHotEncoder)
        onehot_cols = [
            'gender', 'total_meds_up', 'A1Cresult', 'total_prior_visits',
            'race', 'diabetesMed', 'total_meds_steady', 'max_glu_serum',
            'total_meds_down', 'change'
        ]

        # Pipeline de pré-processamento (primeiro faz o imputation, depois o encoding dependendo do tipo de coluna)
        self.preprocessor = ColumnTransformer(
            transformers=[
                # Colunas categóricas com poucas categorias
                ('onehot', Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='missing')),
                    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
                ]), onehot_cols),

                # Colunas categóricas com muitas categorias ordinais
                ('ordinal', Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='missing')),
                    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
                ]), ordinal_cols),

                # Colunas categóricas com muitas categorias não-ordinais
                ('target', Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='missing')),
                    ('encoder', TargetEncoder())
                ]), target_cols),

                # Colunas numéricas
                ('num', Pipeline([
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler())
                ]), numerical_cols)
            ],
            remainder='drop'
        ).set_output(transform="pandas") # Retorna um df do pandas em vez de um array numpy (alguns modelos precisam dos nomes das features)

    # def __impute_data(self, df):


    def __print_prediction_info(self, name, y_pred, cv_scores):
        print(f"\n{'-'*30}")
        print(f"Modelo: {name}")
        print(f"Acurácia média do Cross Evaluation: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        print(f"Acurácia da Predição: {metrics.accuracy_score(self.y_test_encoded, y_pred):.4f}")
        print(f"\n\n{metrics.classification_report(self.y_test_encoded, y_pred, target_names=self.label_encoder.classes_)}")

    def predict(self, models_dict, do_print_info=False):
        predictions_dict = {}

        for name, model in models_dict.items():
            pipeline = Pipeline([
                ('preprocessor', self.preprocessor),
                ('classifier', model)
            ])

            # Cross-validation
            cv_scores = cross_val_score(pipeline, self.X_train, self.y_train_encoded, cv=5)

            pipeline.fit(self.X_train, self.y_train_encoded)
            y_pred = pipeline.predict(self.X_test)

            if do_print_info:
                self.__print_prediction_info(name, y_pred, cv_scores)

            predictions_dict[name] = {'pipeline': pipeline, 'prediction': y_pred, 'accuracy': metrics.accuracy_score(self.y_test_encoded, y_pred)}

        return predictions_dict

#### How to use it

| # | Method                       | Description                                |
|---|------------------------------|--------------------------------------------|
| 1 | `DiabeticDataTrainer(df_raw)`           | Receives the *raw* dataframe, cleans and processes it automatically                     |
| 2 | `predict(models_dict)` | Receives a dict `modelName: model`, predicts df_raw with each and prints results                         |

### Setting Up Data

In [8]:
df_diabetic_data = pd.read_csv(os.path.join('..','data', 'raw','diabetic_data.csv'))

trainer = DiabeticDataTrainer(df_diabetic_data)

C:\Users\edubi\AppData\Local\Temp\ipykernel_19688\3047505783.py:93: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col] = df[col].fillna(df[col].mode()[0], inplace=True)
C:\Users\edubi\AppData\Local\Temp\ipykernel_19688\3047505783.py:93: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained

## INITIAL ANALYSIS

### Testing PCA

Métodos de maior acurácia antes do PCA:

* random forest: ~0.59
* extra trees: ~0.59
* xgboost: ~0.59

Acurácia média antes do PCA: ~0.50

In [ ]:
models_pca_test = {
    "SGD": SGDClassifier(max_iter=2000, class_weight='balanced'),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced'),
    "KNN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', max_depth=10),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight='balanced'),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, class_weight='balanced'),
    "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(eval_metric='mlogloss', random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, auto_class_weights='Balanced', random_state=42),
    "LDA": LinearDiscriminantAnalysis()
}

trainer.predict(models_pca_test, do_print_info=True)


------------------------------
Modelo: SGD
Acurácia média do Cross Evaluation: 0.5445 (+/- 0.0495)
Acurácia da Predição: 0.5650


              precision    recall  f1-score   support

         <30       0.23      0.12      0.16      2229
         >30       0.48      0.40      0.44      7025
          NO       0.63      0.76      0.69     10854

    accuracy                           0.57     20108
   macro avg       0.45      0.43      0.43     20108
weighted avg       0.53      0.57      0.54     20108


------------------------------
Modelo: Logistic Regression
Acurácia média do Cross Evaluation: 0.4993 (+/- 0.0037)
Acurácia da Predição: 0.4988


              precision    recall  f1-score   support

         <30       0.19      0.42      0.26      2229
         >30       0.46      0.36      0.40      7025
          NO       0.68      0.60      0.64     10854

    accuracy                           0.50     20108
   macro avg       0.44      0.46      0.43     20108
weighted avg   

{'SGD': {'prediction': array([2, 2, 2, ..., 2, 2, 2]),
  'accuracy': 0.5650487368211657},
 'Logistic Regression': {'prediction': array([0, 2, 2, ..., 1, 2, 2]),
  'accuracy': 0.49875671374577285},
 'KNN': {'prediction': array([1, 1, 2, ..., 2, 2, 1]),
  'accuracy': 0.509150586831112},
 'Naive Bayes': {'prediction': array([2, 2, 2, ..., 2, 2, 2]),
  'accuracy': 0.534364432066839},
 'Decision Tree': {'prediction': array([0, 1, 1, ..., 1, 2, 2]),
  'accuracy': 0.452506465088522},
 'Random Forest': {'prediction': array([2, 2, 2, ..., 2, 2, 2]),
  'accuracy': 0.582405012930177},
 'Extra Trees': {'prediction': array([2, 2, 2, ..., 2, 2, 2]),
  'accuracy': 0.5789238114183409},
 'LightGBM': {'prediction': array([2, 1, 1, ..., 1, 2, 2]),
  'accuracy': 0.5075094489755321},
 'XGBoost': {'prediction': array([2, 2, 2, ..., 2, 2, 2]),
  'accuracy': 0.5883727869504675},
 'CatBoost': {'prediction': array([[2],
         [2],
         [2],
         ...,
         [1],
         [2],
         [2]]),
  'acc

Nova média: ~0.54

### Testing Advanced Models

In [ ]:
trainer.predict({"SVM (Exponential/RBF Kernel)": SVC(kernel='rbf', probability=True, class_weight='balanced', verbose=True)}, do_print_info=True)

[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]
------------------------------
Modelo: SVM (Exponential/RBF Kernel)
Acurácia média do Cross Evaluation: 0.5033 (+/- 0.0137)
Acurácia da Predição: 0.5038


              precision    recall  f1-score   support

         <30       0.19      0.36      0.25      2272
         >30       0.45      0.32      0.37      7109
          NO       0.65      0.66      0.65     10973

    accuracy                           0.50     20354
   macro avg       0.43      0.44      0.42     20354
weighted avg       0.53      0.50      0.51     20354



{'SVM (Exponential/RBF Kernel)': {'prediction': array([2, 2, 0, ..., 2, 2, 1], shape=(20354,)),
  'accuracy': 0.5038321705807213}}

### Testing the Effects of Imputation

In [ ]:
# Before imputation

randomforests_test = {
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1),
}

results = trainer.predict(randomforests_test, do_print_info=True)


------------------------------
Modelo: Random Forest
Acurácia média do Cross Evaluation: 0.5810 (+/- 0.0053)
Acurácia da Predição: 0.5799


              precision    recall  f1-score   support

         <30       0.59      0.01      0.03      2272
         >30       0.49      0.40      0.44      7109
          NO       0.62      0.81      0.70     10973

    accuracy                           0.58     20354
   macro avg       0.56      0.41      0.39     20354
weighted avg       0.57      0.58      0.54     20354



In [9]:
# After imputation

randomforests_test = {
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1),
}

results = trainer.predict(randomforests_test, do_print_info=True)


------------------------------
Modelo: Random Forest
Acurácia média do Cross Evaluation: 0.5771 (+/- 0.0066)
Acurácia da Predição: 0.5813


              precision    recall  f1-score   support

         <30       0.55      0.02      0.03      2272
         >30       0.49      0.41      0.45      7109
          NO       0.62      0.81      0.70     10973

    accuracy                           0.58     20354
   macro avg       0.55      0.41      0.39     20354
weighted avg       0.57      0.58      0.54     20354



In [12]:
results['Random Forest']['pipeline']

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differen

In [55]:
# Saving the pipeline

file_path = os.path.join('..','data', 'pickled','randomforest_pipeline.pkl')
pickle(results['Random Forest']['pipeline'], file_path)

In [27]:
resultados = results['Random Forest']['pipeline'].predict(trainer.X_test)